In [1]:
import torch

# your journey starts with one step
inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your     (x^1)
    [0.55, 0.87, 0.66], # journey  (x^2)
    [0.57, 0.85, 0.64], # starts   (x^3)
    [0.22, 0.58, 0.33], # with     (x^4)
    [0.77, 0.25, 0.10], # one      (x^5)
    [0.05, 0.80, 0.55]  # step     (x^6)
] )

In [2]:
import torch.nn as nn

class SelfAttentionOwnClassUpdated(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
    def forward(self,x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        
        attention_score = queries @ keys.T
        
        d_k = keys.shape[-1]
        attention_weight = torch.softmax(
            attention_score/d_k**0.5, dim=-1
        )
        context_vector = attention_weight @ values
        return context_vector

In [3]:
d_in = inputs.shape[-1]
d_out = 2 
print(d_in, d_out)

3 2


In [4]:
selfAtten = SelfAttentionOwnClassUpdated(d_in, d_out)

In [5]:
queries = selfAtten.W_query(inputs)
keys = selfAtten.W_key(inputs)

attention_score_without_mask = queries @ keys.T
attention_score_without_mask

tensor([[ 0.0111,  0.0613,  0.0623,  0.0338,  0.0635,  0.0259],
        [ 0.0062,  0.0830,  0.0856,  0.0437,  0.1085,  0.0215],
        [ 0.0095,  0.0863,  0.0886,  0.0462,  0.1050,  0.0270],
        [-0.0073,  0.0329,  0.0353,  0.0151,  0.0679, -0.0066],
        [ 0.0663,  0.1220,  0.1180,  0.0774,  0.0132,  0.1199],
        [-0.0385,  0.0042,  0.0097, -0.0067,  0.1049, -0.0589]],
       grad_fn=<MmBackward0>)

###### Without Masking

In [6]:
d_k = keys.shape[-1]
attention_weight_without_mask = torch.softmax(
    attention_score_without_mask / d_k**0.5, dim=-1
)
attention_weight_without_mask

tensor([[0.1629, 0.1688, 0.1689, 0.1656, 0.1691, 0.1646],
        [0.1606, 0.1696, 0.1699, 0.1649, 0.1727, 0.1623],
        [0.1607, 0.1697, 0.1700, 0.1649, 0.1720, 0.1627],
        [0.1631, 0.1678, 0.1681, 0.1657, 0.1720, 0.1632],
        [0.1643, 0.1709, 0.1704, 0.1656, 0.1582, 0.1706],
        [0.1618, 0.1668, 0.1674, 0.1655, 0.1791, 0.1595]],
       grad_fn=<SoftmaxBackward0>)

###### Using Masking 

In [9]:
attention_score = queries @ keys.T
attention_score

tensor([[ 0.0111,  0.0613,  0.0623,  0.0338,  0.0635,  0.0259],
        [ 0.0062,  0.0830,  0.0856,  0.0437,  0.1085,  0.0215],
        [ 0.0095,  0.0863,  0.0886,  0.0462,  0.1050,  0.0270],
        [-0.0073,  0.0329,  0.0353,  0.0151,  0.0679, -0.0066],
        [ 0.0663,  0.1220,  0.1180,  0.0774,  0.0132,  0.1199],
        [-0.0385,  0.0042,  0.0097, -0.0067,  0.1049, -0.0589]],
       grad_fn=<MmBackward0>)

In [15]:
context_length = attention_score.shape[0]   # (6,6) -> 6

mask = torch.triu( torch.ones(context_length, context_length) , diagonal=1 )
mask


tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])

In [17]:
attention_score_masked = attention_score.masked_fill(mask.bool(), -torch.inf)
attention_score_masked

tensor([[ 0.0111,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.0062,  0.0830,    -inf,    -inf,    -inf,    -inf],
        [ 0.0095,  0.0863,  0.0886,    -inf,    -inf,    -inf],
        [-0.0073,  0.0329,  0.0353,  0.0151,    -inf,    -inf],
        [ 0.0663,  0.1220,  0.1180,  0.0774,  0.0132,    -inf],
        [-0.0385,  0.0042,  0.0097, -0.0067,  0.1049, -0.0589]],
       grad_fn=<MaskedFillBackward0>)

In [18]:
attention_weight_masked = torch.softmax(
    attention_score_masked / d_k**0.5, dim=-1
)
attention_weight_masked

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4864, 0.5136, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3212, 0.3391, 0.3397, 0.0000, 0.0000, 0.0000],
        [0.2454, 0.2525, 0.2529, 0.2493, 0.0000, 0.0000],
        [0.1981, 0.2060, 0.2055, 0.1996, 0.1908, 0.0000],
        [0.1618, 0.1668, 0.1674, 0.1655, 0.1791, 0.1595]],
       grad_fn=<SoftmaxBackward0>)

# Use Dropout

In [19]:
torch.manual_seed(123)
dropout = nn.Dropout(0.5)

dropout(attention_weight_masked)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6424, 0.6782, 0.6794, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5049, 0.5057, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4121, 0.0000, 0.3993, 0.0000, 0.0000],
        [0.0000, 0.3335, 0.3348, 0.3310, 0.3581, 0.0000]],
       grad_fn=<MulBackward0>)